# Lab 3 · Faltantes: ¿imputar o no imputar?
*Desaprendiendo sobre... Limpieza de datos · Estudiante_18 (sector Servicios)*

> **Objetivo.** En este dataset solo falta una cosa: **152 puntuaciones de satisfacción**. Es poco, pero nos sirve para entender lo importante: **por qué** faltan decide **qué hacer**. Y como en la vida real nunca sabemos la verdad, vamos a **borrar datos a propósito** para comprobar qué método acierta.

In [ ]:
# ── Preparación (ejecutar una vez) ───────────────────────────────────────────
# En Colab/Linux usamos binarios precompilados: los paquetes se instalan en segundos.
if (Sys.info()[["sysname"]] == "Linux")
  options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

paquetes <- c("tidyverse", "mice")
faltan <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltan)) install.packages(faltan)
suppressPackageStartupMessages({
library(tidyverse)
})

# Los datos: si no están en la carpeta, se descargan
BASE_URL <- "https://raw.githubusercontent.com/vicentcorrecher/lab-limpieza-datos/main/datos"
obtener <- function(f) {
  ruta <- file.path("datos", f)
  if (!file.exists(ruta)) {
    dir.create(dirname(ruta), recursive = TRUE, showWarnings = FALSE)
    download.file(paste0(BASE_URL, "/", f), ruta, quiet = TRUE, mode = "wb")
  }
}
for (f in c(
  "funciones_labs.R",
  "estudiante_18/operaciones.csv",
  "estudiante_18/clientes.csv",
  "estudiante_18/productos_servicios.csv",
  "estudiante_18/sedes.csv",
  "estudiante_18/campanas.csv"
)) obtener(f)
source("datos/funciones_labs.R")

In [ ]:
d <- cargar_e18()
l <- limpiar_e18(d)                     # limpieza del Lab 2 (solo duplicados + banderas)
ops <- l$operaciones |>
  left_join(select(l$clientes, cliente_id, segmento), by = "cliente_id") |>
  mutate(anyo = year(fecha))

cat("Satisfacciones vacías:", sum(is.na(ops$satisfaccion)), "de", nrow(ops), "(", round(100 * mean(is.na(ops$satisfaccion)), 2), "%)\n")

## 1. ¿Dónde faltan? ¿Al azar o con patrón?

Comparamos el % de vacías entre grupos. Si faltara **al azar** (MCAR), todos los grupos tendrían un % parecido.

In [ ]:
vars <- c("canal", "segmento", "sede_id", "categoria", "incidencia", "anyo")

perfil <- map_dfr(vars, \(v) ops |>
  group_by(grupo = as.character(.data[[v]])) |>
  summarise(operaciones = n(), pct_vacias = round(100 * mean(is.na(satisfaccion)), 2), .groups = "drop") |>
  mutate(variable = v, .before = 1))
perfil

In [ ]:
# Prueba formal (chi-cuadrado): p pequeño = hay relación entre «faltar» y esa variable
pruebas <- map_dfr(vars, \(v) tibble(
  variable = v,
  p_valor = suppressWarnings(chisq.test(table(ops[[v]], is.na(ops$satisfaccion)))$p.value))) |>
  mutate(p_valor = round(p_valor, 3), parece_aleatorio = p_valor > 0.05)
pruebas

Una variable (la **sede**) sale «significativa». Pero hemos hecho 6 pruebas: con un umbral de 0,05 es normal que **una de cada 20** salga significativa por pura casualidad.

> ### Para pensar
> 1. Con esta evidencia, ¿diríais MCAR, MAR o MNAR? ¿Qué os haría cambiar de opinión?
> 2. ¿Por qué es **peligroso** que la satisfacción falte más cuando el cliente está descontento? (pista: ¿quién deja la encuesta en blanco?)
> 3. El % de vacías es del 0,8 %. ¿Tiene sentido gastar tiempo en esto?

## 2. Cuatro formas de tratar los vacíos

In [ ]:
sat <- ops$satisfaccion

# a) Ignorar las vacías (casos completos)
est_a <- sat[!is.na(sat)]

# b) Imputar la media global
est_b <- if_else(is.na(sat), mean(sat, na.rm = TRUE), sat)

# c) Imputar la media por sede y categoría de servicio
est_c <- ops |> group_by(sede_id, categoria) |>
  mutate(x = if_else(is.na(satisfaccion), mean(satisfaccion, na.rm = TRUE), satisfaccion)) |> ungroup() |> pull(x)

# d) Imputación múltiple con mice (PMM): 3 versiones plausibles
dat <- ops |> select(satisfaccion, unidades, descuento_pct, duracion_min, canal, categoria, sede_id, incidencia) |>
  mutate(across(c(canal, categoria, sede_id, incidencia), factor))
set.seed(2026)
imp <- mice::mice(dat, m = 3, maxit = 3, method = "pmm", printFlag = FALSE)
completos <- map(1:3, \(i) mice::complete(imp, i)$satisfaccion)

tibble(estrategia = c("Ignorar vacías", "Media global", "Media por sede y categoría", "mice (PMM, m = 3)"),
       satisfaccion_media = c(mean(est_a), mean(est_b), mean(est_c), mean(map_dbl(completos, mean))),
       desviacion_tipica  = c(sd(est_a), sd(est_b), sd(est_c), mean(map_dbl(completos, sd)))) |>
  mutate(across(where(is.numeric), \(x) round(x, 4)))

Con solo un 0,8 % de vacías, **los cuatro métodos dan casi lo mismo**. Fíjate en la desviación típica: la imputación por la media la **encoge** un poquito (inventa valores «promedio»).

## 3. El experimento: ¿qué pasa si faltaran muchos más?

Aquí está el truco. Tomamos solo las filas **con** satisfacción (conocemos la verdad), **borramos una parte a propósito** de dos maneras y vemos qué método se acerca más a la verdad:

- **MCAR:** se borra al azar.
- **MNAR:** se borran sobre todo las puntuaciones **bajas** (los descontentos no contestan).

In [ ]:
# >>> TOCA AQUÍ: ¿qué % de datos borramos? (prueba 0.05, 0.15, 0.40) <<<
PCT_BORRADO <- 0.15

base <- ops |> filter(!is.na(satisfaccion))
verdad_media <- mean(base$satisfaccion); verdad_sd <- sd(base$satisfaccion)

simular <- function(mecanismo, pct, semilla = 1) {
  set.seed(semilla)
  peso <- if (mecanismo == "MCAR") rep(1, nrow(base)) else (11 - base$satisfaccion)^3
  x <- base
  x$satisfaccion[sample(nrow(base), round(pct * nrow(base)), prob = peso)] <- NA
  g <- mean(x$satisfaccion, na.rm = TRUE)
  imp_global <- if_else(is.na(x$satisfaccion), g, x$satisfaccion)
  imp_grupo <- x |> group_by(sede_id, categoria) |>
    mutate(v = if_else(is.na(satisfaccion), mean(satisfaccion, na.rm = TRUE), satisfaccion)) |> ungroup() |> pull(v)
  tibble(mecanismo, metodo = c("Ignorar vacías", "Media global", "Media por sede y categoría"),
         media = c(g, mean(imp_global), mean(imp_grupo)),
         desviacion = c(sd(x$satisfaccion, na.rm = TRUE), sd(imp_global), sd(imp_grupo)))
}

resultados <- bind_rows(simular("MCAR", PCT_BORRADO), simular("MNAR", PCT_BORRADO)) |>
  mutate(sesgo_media = media - verdad_media, sesgo_desviacion = desviacion - verdad_sd)
resultados |> mutate(across(where(is.numeric), \(x) round(x, 3)))

In [ ]:
ggplot(resultados, aes(metodo, sesgo_media, fill = mecanismo)) +
  geom_col(position = "dodge") +
  geom_hline(yintercept = 0) +
  scale_fill_manual(values = c(MCAR = "#2BB5A6", MNAR = "#F2B84B")) +
  labs(title = paste0("Error en la satisfacción media si falta el ", PCT_BORRADO * 100, " % de los datos"),
       subtitle = "0 = acierta la verdad. MCAR = al azar · MNAR = faltan las puntuaciones bajas",
       x = NULL, y = "media estimada − media real", fill = NULL) +
  theme_minimal(base_size = 13)

> ### Para pensar
> 1. Con MCAR, los métodos apenas se equivocan. Con MNAR, **todos se equivocan igual**. ¿Qué conclusión sacas: «hay que elegir mejor el método» o «ningún método arregla un dato que falta por una razón»?
> 2. Fíjate en la columna `sesgo_desviacion`: ¿qué método hace que los clientes parezcan **más parecidos** entre sí de lo que son?
> 3. Si la satisfacción es un KPI del cuadro de mando, ¿qué **avisarías** al director cuando falta mucha?

### Toca esto

Cambia `PCT_BORRADO` a `0.40` y ejecuta de nuevo la celda del experimento y el gráfico. ¿A partir de qué porcentaje deja de ser fiable el KPI?

## 4. Conclusión de este dataset

Con un 0,8 % de vacías, que parecen casi aleatorias, **lo más honesto y simple es no imputar**: se calcula la media con los datos disponibles y se anota «n = 18.848 respuestas». Imputar tendría sentido si faltaran muchas más, o si fuera una variable que necesitas sí o sí para un modelo.